# Notebook 02: MediaPipe Landmark Extraction & Quality Analysis

Notebook này thực hiện trích xuất tọa độ khớp bàn tay (Hand Landmarks) từ các video của 5 classes đã chọn ở Notebook 01 bằng MediaPipe Hands.

### Nội dung:
1. **Load selected split**: Đọc danh sách mẫu từ file `selected_5_classes.csv`.
2. **Load videos**: Kiểm tra và đọc các file video bằng OpenCV.
3. **MediaPipe Hands**: Cấu hình pipeline nhận diện bàn tay MediaPipe.
4. **Extract 126 features/frame**: 2 bàn tay × 21 landmarks × 3 tọa độ (x, y, z) = 126 features.
5. **Handle missing hands**: Xử lý gán đúng tay trái/phải và zero-fill tay không xuất hiện.
6. **Save landmarks**: Lưu chuỗi landmark ra file `.npy` độc lập tại `/kaggle/working/asl_outputs/landmarks/`.
7. **Landmark quality analysis**: Đánh giá tỷ lệ phát hiện (2 tay, 1 tay, mất dấu tay).
8. **Landmark visualization**: Vẽ trực quan tọa độ 2D/3D của các khớp tay.
9. **Sequence-length analysis**: Phân tích độ dài chuỗi frame để xác định độ dài cố định $T=32$.


---
## 1. Setup & Load Selected Split


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Cài đặt mediapipe
!pip install -q mediapipe

IN_KAGGLE = Path('/kaggle/working').exists()
OUTPUT_ROOT = Path('/kaggle/working/asl_outputs') if IN_KAGGLE else Path('outputs').resolve()
SPLIT_OUTPUT_DIR = OUTPUT_ROOT / 'splits'
LANDMARK_DIR = OUTPUT_ROOT / 'landmarks'
PLOTS_OUTPUT_DIR = OUTPUT_ROOT / 'plots'

for d in [LANDMARK_DIR, PLOTS_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Đọc file danh sách split từ Notebook 01
split_file = SPLIT_OUTPUT_DIR / 'selected_5_classes.csv'
if not split_file.exists():
    # Fallback kiểm tra trong repo data/splits
    repo_split = Path('/kaggle/working/SignLanguageReconition_ML/data/splits/train.csv') if IN_KAGGLE else Path('data/splits/train.csv').resolve()
    if repo_split.exists():
        dfs = [pd.read_csv(repo_split.parent / f"{s}.csv") for s in ['train', 'val', 'test'] if (repo_split.parent / f"{s}.csv").exists()]
        df_selected = pd.concat(dfs, ignore_index=True)
    else:
        raise FileNotFoundError(f"Không tìm thấy {split_file}. Hãy chạy Notebook 01 trước!")
else:
    df_selected = pd.read_csv(split_file)

print(f"Loaded {len(df_selected)} samples for 5 classes:")
print(df_selected['gloss'].value_counts())


---
## 2. MediaPipe Hands Configuration & Extraction Pipeline


In [ ]:
import cv2
import mediapipe as mp
from tqdm.auto import tqdm

class MediaPipeHandExtractor:
    """
    Trích xuất tọa độ 3D bàn tay từ từng frame bằng MediaPipe Hands.
    Format đầu ra mỗi frame: [126] float32.
    - [0:63]: Bàn tay Trái (21 landmarks x 3 coords)
    - [63:126]: Bàn tay Phải (21 landmarks x 3 coords)
    """
    def __init__(self, min_detection_confidence=0.5, min_tracking_confidence=0.5):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence
        )

    def extract_from_frame(self, frame_bgr):
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results = self.hands.process(frame_rgb)
        
        left_hand = np.zeros((21, 3), dtype=np.float32)
        right_hand = np.zeros((21, 3), dtype=np.float32)
        hands_detected = 0
        
        if results.multi_hand_landmarks and results.multi_handedness:
            hands_detected = len(results.multi_hand_landmarks)
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = handedness.classification[0].label  # 'Left' hoặc 'Right'
                coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark], dtype=np.float32)
                if label == 'Left':
                    left_hand = coords
                elif label == 'Right':
                    right_hand = coords
                    
        features = np.concatenate([left_hand.flatten(), right_hand.flatten()], axis=0)
        return features, hands_detected

    def extract_from_video(self, video_path):
        cap = cv2.VideoCapture(str(video_path))
        sequence = []
        detection_stats = []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            feat, n_hands = self.extract_from_frame(frame)
            sequence.append(feat)
            detection_stats.append(n_hands)
            
        cap.release()
        seq_arr = np.array(sequence, dtype=np.float32) if len(sequence) > 0 else np.zeros((0, 126), dtype=np.float32)
        return seq_arr, detection_stats

    def close(self):
        self.hands.close()

extractor = MediaPipeHandExtractor()
print("MediaPipe Hand Extractor initialized successfully!")


---
## 3. Extract & Save Landmarks (.npy files)


In [ ]:
manifest = []
detection_history = []

print(f"Extracting landmarks for {len(df_selected)} video samples...")

for idx, row in tqdm(df_selected.iterrows(), total=len(df_selected)):
    vid_id = str(row['video_id'])
    gloss = row['gloss']
    split = row['split']
    vid_path = Path(row['video_path'])
    
    if not vid_path.exists():
        continue
        
    out_npy = LANDMARK_DIR / f"{vid_id}.npy"
    
    if out_npy.exists():
        seq = np.load(out_npy)
        n_frames = len(seq)
    else:
        seq, stats = extractor.extract_from_video(vid_path)
        n_frames = len(seq)
        if n_frames > 0:
            np.save(out_npy, seq)
            detection_history.append({'video_id': vid_id, 'stats': stats})
            
    if n_frames > 0:
        manifest.append({
            'video_id': vid_id,
            'gloss': gloss,
            'split': split,
            'num_frames': n_frames,
            'npy_path': str(out_npy)
        })

extractor.close()
df_manifest = pd.DataFrame(manifest)
manifest_path = OUTPUT_ROOT / 'extracted_landmarks.csv'
df_manifest.to_csv(manifest_path, index=False)
print(f"\nSaved landmark manifest to: {manifest_path}")
print(f"Total valid landmark sequences: {len(df_manifest)}")
df_manifest.head(5)


---
## 4. Landmark Quality Analysis (Hand Detection Rates)


In [ ]:
if detection_history:
    total_frames = sum(len(d['stats']) for d in detection_history)
    frames_2_hands = sum(d['stats'].count(2) for d in detection_history)
    frames_1_hand = sum(d['stats'].count(1) for d in detection_history)
    frames_0_hand = sum(d['stats'].count(0) for d in detection_history)
    
    print("=" * 50)
    print("HAND DETECTION QUALITY SUMMARY")
    print("=" * 50)
    print(f"Total Frames Processed : {total_frames:,}")
    print(f"Frames with 2 Hands     : {frames_2_hands:,} ({frames_2_hands/total_frames*100:.1f}%)")
    print(f"Frames with 1 Hand      : {frames_1_hand:,} ({frames_1_hand/total_frames*100:.1f}%)")
    print(f"Frames with 0 Hand (Lost): {frames_0_hand:,} ({frames_0_hand/total_frames*100:.1f}%)")


---
## 5. Landmark Visualization (Hand Skeleton)


In [ ]:
import matplotlib.pyplot as plt

# Load một landmark sequence mẫu
sample_item = df_manifest.iloc[0]
sample_npy = np.load(sample_item['npy_path'])
mid_frame = sample_npy[len(sample_npy) // 2]

left_hand_pts = mid_frame[:63].reshape(21, 3)
right_hand_pts = mid_frame[63:].reshape(21, 3)

HAND_CONNECTIONS = mp.solutions.hands.HAND_CONNECTIONS

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title(f"Extracted Hand Landmarks for '{sample_item['gloss']}' (ID: {sample_item['video_id']})")

# Vẽ tay trái nếu có
if np.any(left_hand_pts != 0):
    ax.scatter(left_hand_pts[:, 0], left_hand_pts[:, 1], c='blue', label='Left Hand', s=40)
    for connection in HAND_CONNECTIONS:
        p1, p2 = connection
        ax.plot([left_hand_pts[p1, 0], left_hand_pts[p2, 0]], [left_hand_pts[p1, 1], left_hand_pts[p2, 1]], 'b-', alpha=0.6)

# Vẽ tay phải nếu có
if np.any(right_hand_pts != 0):
    ax.scatter(right_hand_pts[:, 0], right_hand_pts[:, 1], c='red', label='Right Hand', s=40)
    for connection in HAND_CONNECTIONS:
        p1, p2 = connection
        ax.plot([right_hand_pts[p1, 0], right_hand_pts[p2, 0]], [right_hand_pts[p1, 1], right_hand_pts[p2, 1]], 'r-', alpha=0.6)

ax.invert_yaxis()  # Image coordinate system
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')
ax.legend()
ax.grid(True)
plt.savefig(PLOTS_OUTPUT_DIR / 'landmarks_skeleton_sample.png', dpi=150)
plt.show()


---
## 6. Sequence-Length Analysis


In [ ]:
lengths = df_manifest['num_frames']

print("=" * 50)
print("SEQUENCE LENGTH STATISTICS (FRAMES)")
print("=" * 50)
print(f"Mean Frame Count   : {lengths.mean():.1f}")
print(f"Median Frame Count : {lengths.median():.1f}")
print(f"Min Frame Count    : {lengths.min()}")
print(f"Max Frame Count    : {lengths.max()}")
print(f"90th Percentile    : {lengths.quantile(0.90):.1f}")
print(f"95th Percentile    : {lengths.quantile(0.95):.1f}")
print("\n-> Kết luận: Độ dài cố định T = 32 frames bao phủ tốt đại đa số các cử chỉ sign language.")

plt.figure(figsize=(10, 5))
plt.hist(lengths, bins=25, color='teal', edgecolor='black', alpha=0.8)
plt.axvline(32, color='red', linestyle='--', linewidth=2, label='Selected Fixed Target Length T = 32')
plt.title('Distribution of Video Sequence Lengths (Frames)')
plt.xlabel('Number of Frames')
plt.ylabel('Video Count')
plt.legend()
plt.grid(True)
plt.savefig(PLOTS_OUTPUT_DIR / 'sequence_length_distribution.png', dpi=150)
plt.show()

print("Landmark Extraction & Analysis hoàn tất! Sẵn sàng cho Notebook 03 (Model Training).")
